# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 13 — Is the association with degree an artefact of the input encoding?

The mechanism proposed for the central result is that message passing aggregates over
neighbourhoods, so gradient magnitude accumulates on atoms with more neighbours and the
resulting ranking approximates a degree ranking. There is a simpler explanation that has not
been ruled out.

Atom degree is one of the twenty-four atom features. Integrated Gradients is computed from a
zero baseline and aggregated as an absolute sum over feature dimensions, so the contribution
of the degree feature to atom $i$ is approximately its value times the mean gradient along
that dimension, and therefore monotone in degree by construction. The one-hot dimensions take
values in $\{0,1\}$ while degree runs from one to four, so that dimension is also the
largest in magnitude. Under this account the association would have nothing to do with
message passing.

The existing controls do not separate the two. The random weighting of atom features, which
uses no message passing, falls below chance at $0.972$, but it also uses no gradients, so it
tests a different thing.

This notebook removes degree from the input encoding and repeats the measurement. It also
varies the Integrated Gradients baseline, since a zero baseline is off-manifold for one-hot
inputs.

**Contents**

1. Environment and a configurable encoding
2. Data and evaluation set
3. Pre-registration
4. Untrained networks under three encodings
5. Twenty trained models without the degree feature
6. Sensitivity to the attribution baseline
7. What the encoding costs in accuracy
8. Summary

*Approximate runtime: two to three hours on a T4. Each stage checkpoints.*

## 1. Environment and a configurable encoding

In [1]:
!pip install -q rdkit
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 44.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
CKPT = f"{WORK}/ckpt_part13"
for d in (R, EXPL, MODELS, CKPT): os.makedirs(d, exist_ok=True)
print("checkpoints:", CKPT)

Mounted at /content/drive
checkpoints: /content/drive/MyDrive/ames_pipeline/ckpt_part13


The definitions below are those used throughout, with one change: the atom encoding takes a
list of feature indices to drop. Dropping nothing reproduces the encoding used in every
earlier part, which the assertion at the end of the cell checks.

In [3]:
%%writefile /content/ames13.py
"""Shared definitions, with an atom encoding that can omit chosen features."""
import random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator as rfg
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")
BOND_DIM = 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']

# index of each scalar feature in the full 24-dimensional vector
IDX_DEGREE   = len(ATOM_LIST) + len(HYBRIDIZATION)          # 19
IDX_CHARGE   = IDX_DEGREE + 1                                # 20
IDX_AROMATIC = IDX_DEGREE + 2                                # 21
IDX_NUM_H    = IDX_DEGREE + 3                                # 22
IDX_IN_RING  = IDX_DEGREE + 4                                # 23
FULL_ATOM_DIM = IDX_IN_RING + 1                              # 24

ENCODINGS = {
    "full":            [],
    "no_degree":       [IDX_DEGREE],
    "no_degree_no_h":  [IDX_DEGREE, IDX_NUM_H],
}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features_full(a):
    f  = one_hot(a.GetSymbol(), ATOM_LIST) + one_hot(str(a.GetHybridization()), HYBRIDIZATION)
    f += [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
          a.GetTotalNumHs(), int(a.IsInRing())]
    return f

def atom_features(a, drop=()):
    f = atom_features_full(a)
    return [v for i, v in enumerate(f) if i not in set(drop)]

def atom_dim(drop=()):
    return FULL_ATOM_DIM - len(set(drop))

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_data(smiles, label, drop=()):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.tensor([atom_features(a, drop) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d, drop=()):
    g = [smiles_to_data(r["canonical"], float(r["label"]), drop) for _, r in d.iterrows()]
    return [x for x in g if x is not None]

class GNN(nn.Module):
    def __init__(self, kind, in_dim=FULL_ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn": conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else: raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h)); h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4,
              use_edge_attr=True, in_dim=FULL_ATOM_DIM):
    return GNN(kind, in_dim=in_dim, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(DEVICE)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(model(d)).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_gnn(kind, params, seed, tr_set, va_set, pos_weight, in_dim=FULL_ATOM_DIM,
              max_epochs=250, patience=30, use_edge_attr=True):
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr, in_dim=in_dim)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"],
                           weight_decay=params["weight_decay"])
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tl = DataLoader(tr_set, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    vl = DataLoader(va_set, batch_size=256)
    best, best_state, waited = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for d in tl:
            d = d.to(DEVICE); opt.zero_grad()
            crit(model(d), d.y.view(-1)).backward(); opt.step()
        y, p = evaluate(model, vl)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
        if auc > best:
            best, waited = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best

Writing /content/ames13.py


In [4]:
import sys, json, time, glob
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames13 import *
from scipy import stats

_m = Chem.MolFromSmiles("CC(=O)Nc1ccccc1")
_a = _m.GetAtomWithIdx(1)
assert len(atom_features(_a)) == 24 and atom_dim() == 24
assert len(atom_features(_a, ENCODINGS["no_degree"])) == 23
assert atom_features(_a)[IDX_DEGREE] == _a.GetDegree()
assert IDX_DEGREE not in [i for i, _ in enumerate(atom_features(_a, ENCODINGS["no_degree"]))] or True
_full = atom_features(_a); _red = atom_features(_a, ENCODINGS["no_degree"])
assert _red == [v for i, v in enumerate(_full) if i != IDX_DEGREE]
print("encoding check passed; full dimension", atom_dim(),
      "| without degree", atom_dim(ENCODINGS["no_degree"]),
      "| without degree and hydrogens", atom_dim(ENCODINGS["no_degree_no_h"]))

encoding check passed; full dimension 24 | without degree 23 | without degree and hydrogens 22


## 2. Data and evaluation set

In [5]:
SEED, PRIMARY_SPLIT_SEED = 42, 101
N_MODELS, N_UNTRAINED = 20, 500
K_FRAC, N_IG_STEPS, N_TIEBREAK = 0.30, 32, 200
HP = json.load(open(f"{R}/hpo_gnn.json"))["gin"]

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]
assert set(df_test["canonical"]) == set(pd.read_csv(f"{R}/primary_test.csv")["canonical"])
y_train, y_test = df_train["label"].values, df_test["label"].values
POS_W = torch.tensor([(y_train==0).sum()/max((y_train==1).sum(),1)],
                     dtype=torch.float, device=DEVICE)

TOX = {"aromatic_nitro":"[a][N+](=O)[O-]","aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
 "aromatic_nitroso":"[a][NX2]=[OX1]","alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
 "nitrosamine":"[NX3][NX2]=[OX1]","epoxide":"C1OC1","aziridine":"C1NC1",
 "azide":"[NX2]=[NX2+]=[NX1-]","azo_aromatic":"[a][NX2]=[NX2][a]",
 "aliphatic_halide":"[CX4][F,Cl,Br,I]","hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
 "polycyclic_aromatic":"c1ccc2ccccc2c1","aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
 "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]","aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
 "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
 "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]","alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])"}
PAT = {k: Chem.MolFromSmarts(v) for k, v in TOX.items()}

EVAL = []
for smi, lab in zip(df_test["canonical"], y_test):
    if lab != 1: continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0: continue
    t = set()
    for p in PAT.values():
        for mt in mol.GetSubstructMatches(p): t.update(mt)
    if not t or len(t) == mol.GetNumAtoms(): continue
    EVAL.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})
CHANCE = np.array([len(e["tox"])/e["n_atoms"] for e in EVAL]); N_EVAL = len(EVAL)
DEG = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in EVAL]

def overlap(scores, k_frac=K_FRAC, tie_seed=0):
    rng = np.random.default_rng(tie_seed); out = np.full(N_EVAL, np.nan)
    for i, (e, sc) in enumerate(zip(EVAL, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"])/k
    return out

def lift(ov):
    m = ~np.isnan(ov)
    return float(ov[m].mean()/CHANCE[m].mean()) if m.sum() else np.nan

def scored(scores, n=N_TIEBREAK, base=0):
    return np.array([lift(overlap(scores, K_FRAC, base+j)) for j in range(n)])

def rho_with_degree(scores):
    """Mean within-molecule Spearman between an atom ranking and atom degree."""
    v = [stats.spearmanr(s, d).correlation
         if len(set(d)) > 1 and len(set(np.round(np.asarray(s, float), 12))) > 1 else np.nan
         for s, d in zip(scores, DEG)]
    v = np.array(v, dtype=float)
    return float(np.nanmean(v)), float(np.nanmean(v > 0))

DEG_LIFT = scored(DEG, base=4000).mean()
print(f"evaluation set {N_EVAL}; degree baseline {DEG_LIFT:.4f}")

GRAPHS = {k: [smiles_to_data(e["smiles"], 1.0, drop) for e in EVAL]
          for k, drop in ENCODINGS.items()}
for k in ENCODINGS:
    print(f"  {k:16s} atom dimension {GRAPHS[k][0].x.shape[1]}")

evaluation set 237; degree baseline 1.2136
  full             atom dimension 24
  no_degree        atom dimension 23
  no_degree_no_h   atom dimension 22


## 3. Pre-registration

The reading of this experiment is fixed before it is run.

Removing degree from the input encoding does not remove the information: a message-passing
network can still recover a node's degree from the structure of its neighbourhood. What
removal does is close the direct path, in which Integrated Gradients assigns an atom
attribution proportional to the value of its degree feature.

| result for $\rho$(IG, degree), untrained networks | reading |
|---|---|
| stays above $0.30$ | the association is produced by message passing; the mechanism stands |
| falls below $0.10$ | the association was an artefact of the input encoding; the mechanism section is withdrawn and rewritten |
| between the two | both paths contribute; the contribution of each is reported and no single mechanism is claimed |

The comparison quantity is the within-molecule Spearman correlation between the attribution
scores and atom degree, averaged over molecules and over draws. Under the full encoding it is
$0.53$ for the representative model and $0.43$ averaged over twenty trained models; the
untrained figure is $0.685$.

Toxicophore agreement is reported alongside but is not the deciding quantity, since removing
a feature changes what the network can learn as well as how it attributes.

A second encoding, without both degree and hydrogen count, is included because the two are
related through valence; if the association survives the first but not the second, that is
informative rather than contradictory.

## 4. Untrained networks under three encodings

The cleanest test. An untrained network has learned nothing, so any association with degree
comes either from the input encoding or from the architecture acting on the graph.

In [6]:
def grad_scores(model, graphs, method="ig", n_steps=N_IG_STEPS, batch_size=64,
               baseline="zero", ref=None):
    """Attribution scores. baseline='zero' integrates from the origin; 'mean' integrates
    from the dataset-average atom vector, which is on the data manifold for one-hot inputs."""
    was, saved = model.training, [p.requires_grad for p in model.parameters()]
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        batch = batch.to(DEVICE); x0 = batch.x.clone()
        base = torch.zeros_like(x0) if baseline == "zero" else ref.to(DEVICE).expand_as(x0)
        if method == "ig":
            total = torch.zeros_like(x0)
            for al in torch.linspace(1.0/n_steps, 1.0, n_steps):
                xa = (base + al*(x0-base)).detach().requires_grad_(True)
                lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
                total += torch.autograd.grad(lg.sum(), xa)[0]
            attr = (x0-base) * (total/n_steps)
        else:
            xa = x0.clone().detach().requires_grad_(True)
            lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            attr = torch.autograd.grad(lg.sum(), xa)[0]
        sc = attr.abs().sum(dim=1).detach().cpu().numpy(); b = batch.batch.cpu().numpy()
        for gi in range(batch.num_graphs): out.append(sc[b == gi])
    for p, f in zip(model.parameters(), saved): p.requires_grad_(f)
    model.train(was)
    return out

MEAN_REF = {k: torch.stack([g.x for g in GRAPHS[k]]).mean(0, keepdim=True)
            if False else torch.cat([g.x for g in GRAPHS[k]]).mean(0, keepdim=True)
            for k in ENCODINGS}

UN_PATH = f"{R}/table66_encoding_untrained.csv"
if os.path.exists(UN_PATH):
    un = pd.read_csv(UN_PATH); print("loaded from checkpoint")
else:
    rows = []
    for enc, drop in ENCODINGS.items():
        dim = atom_dim(drop); t0 = time.time()
        rhos, poss, lifts = [], [], []
        for j in range(N_UNTRAINED):
            set_seed(97000+j)
            m = build_gnn("gin", HP["hidden"], HP["n_layers"], HP["dropout"],
                          HP.get("residual", True), in_dim=dim)
            sc = grad_scores(m, GRAPHS[enc], "ig")
            r, pos = rho_with_degree(sc)
            rhos.append(r); poss.append(pos); lifts.append(lift(overlap(sc, K_FRAC, 97000+j)))
            del m
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            if (j+1) % 100 == 0:
                print(f"  {enc}: {j+1}/{N_UNTRAINED} ({(time.time()-t0)/60:.1f} min)")
        rows.append({"encoding": enc, "atom_dim": dim, "n_draws": N_UNTRAINED,
                     "rho_mean": float(np.mean(rhos)), "rho_sd": float(np.std(rhos, ddof=1)),
                     "rho_ci_low": float(np.percentile(rhos, 2.5)),
                     "rho_ci_high": float(np.percentile(rhos, 97.5)),
                     "pct_molecules_positive": 100*float(np.mean(poss)),
                     "lift_mean": float(np.mean(lifts)), "lift_sd": float(np.std(lifts, ddof=1)),
                     "minutes": (time.time()-t0)/60})
        np.savez_compressed(f"{EXPL}/encoding_untrained_{enc}.npz",
                            rho=np.array(rhos), lift=np.array(lifts))
        print(f"  {enc}: rho {rows[-1]['rho_mean']:+.3f}, lift {rows[-1]['lift_mean']:.4f}")
    un = pd.DataFrame(rows); un.to_csv(UN_PATH, index=False)
print()
print(un.round(4).to_string(index=False))

  full: 100/500 (1.5 min)
  full: 200/500 (2.9 min)
  full: 300/500 (4.3 min)
  full: 400/500 (5.7 min)
  full: 500/500 (7.1 min)
  full: rho +0.678, lift 1.1175
  no_degree: 100/500 (1.4 min)
  no_degree: 200/500 (2.8 min)
  no_degree: 300/500 (4.2 min)
  no_degree: 400/500 (5.5 min)
  no_degree: 500/500 (6.9 min)
  no_degree: rho +0.481, lift 1.0519
  no_degree_no_h: 100/500 (1.4 min)
  no_degree_no_h: 200/500 (2.8 min)
  no_degree_no_h: 300/500 (4.2 min)
  no_degree_no_h: 400/500 (5.6 min)
  no_degree_no_h: 500/500 (7.0 min)
  no_degree_no_h: rho +0.658, lift 1.1175

      encoding  atom_dim  n_draws  rho_mean  rho_sd  rho_ci_low  rho_ci_high  pct_molecules_positive  lift_mean  lift_sd  minutes
          full        24      500    0.6776  0.1033      0.4396       0.8322                 98.5241     1.1175   0.0607   7.0921
     no_degree        23      500    0.4811  0.1583      0.0894       0.7061                 92.6658     1.0519   0.0725   6.8995
no_degree_no_h        22      500

## 5. Twenty trained models without the degree feature

If the association is produced by message passing it should survive training as well as
initialisation. Twenty models are fitted under each reduced encoding, using the protocol and
the configuration selected for the full encoding, and their test performance is recorded so
that a loss of accuracy cannot be mistaken for a loss of association.

In [7]:
TR_PATH = f"{CKPT}/encoding_models.csv"
meta = pd.read_csv(TR_PATH).to_dict("records") if os.path.exists(TR_PATH) else []
done = {(r["encoding"], int(r["seed"])) for r in meta}
t0 = time.time()
for enc in ["no_degree", "no_degree_no_h"]:
    drop = ENCODINGS[enc]; dim = atom_dim(drop)
    tr_set = build_dataset(df_train, drop); va_set = build_dataset(df_valid, drop)
    te_loader = DataLoader(build_dataset(df_test, drop), batch_size=256)
    for i in range(N_MODELS):
        sd = 98000 + i
        if (enc, sd) in done: continue
        m, va = train_gnn("gin", HP, sd, tr_set, va_set, POS_W, in_dim=dim)
        y, p = evaluate(m, te_loader)
        sc = grad_scores(m, GRAPHS[enc], "ig")
        r, pos = rho_with_degree(sc)
        rec = {"encoding": enc, "seed": sd, "atom_dim": dim, "valid_auc": float(va),
               "test_roc_auc": roc_auc_score(y, p), "rho_ig_degree": r,
               "pct_positive": 100*pos, "lift": scored(sc, n=50, base=sd).mean()}
        torch.save({"state_dict": m.state_dict(), "valid_auc": float(va)},
                   f"{MODELS}/gin_{enc}_seed{sd}.pt")
        np.savez_compressed(f"{EXPL}/scores_ig_{enc}_seed{sd}.npz",
                            **{f"m{j}": s for j, s in enumerate(sc)})
        meta.append(rec); pd.DataFrame(meta).to_csv(TR_PATH, index=False)
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"  {enc} {sd}  test {rec['test_roc_auc']:.4f}  rho {r:+.3f}  "
              f"lift {rec['lift']:.4f}  ({(time.time()-t0)/60:.0f} min)")

meta = pd.DataFrame(meta)
full = pd.read_csv(f"{R}/table59_architecture_models.csv")
full = full[full.arch == "gin"]
rows = [{"encoding": "full", "atom_dim": 24, "n_models": len(full),
         "test_roc_auc": full.test_roc_auc.mean(),
         "rho_mean": full.spearman_ig_degree.mean(),
         "rho_sd": full.spearman_ig_degree.std(ddof=1),
         "lift_mean": full.lift_ig_k30.mean(), "lift_sd": full.lift_ig_k30.std(ddof=1)}]
for enc in ["no_degree", "no_degree_no_h"]:
    s = meta[meta.encoding == enc]
    if not len(s): continue
    rows.append({"encoding": enc, "atom_dim": int(s.atom_dim.iloc[0]), "n_models": len(s),
                 "test_roc_auc": s.test_roc_auc.mean(),
                 "rho_mean": s.rho_ig_degree.mean(), "rho_sd": s.rho_ig_degree.std(ddof=1),
                 "lift_mean": s.lift.mean(), "lift_sd": s.lift.std(ddof=1)})
trained = pd.DataFrame(rows)
trained.to_csv(f"{R}/table67_encoding_trained.csv", index=False)
print()
print(trained.round(4).to_string(index=False))

  no_degree 98000  test 0.7934  rho +0.225  lift 1.2397  (2 min)
  no_degree 98001  test 0.8025  rho -0.054  lift 1.0400  (4 min)
  no_degree 98002  test 0.7840  rho +0.324  lift 1.2197  (7 min)
  no_degree 98003  test 0.7956  rho -0.063  lift 1.0745  (9 min)
  no_degree 98004  test 0.7804  rho +0.222  lift 1.1195  (12 min)
  no_degree 98005  test 0.7744  rho +0.195  lift 1.0166  (13 min)
  no_degree 98006  test 0.7799  rho +0.389  lift 1.2689  (15 min)
  no_degree 98007  test 0.7818  rho +0.197  lift 1.2314  (16 min)
  no_degree 98008  test 0.7884  rho +0.274  lift 1.1885  (19 min)
  no_degree 98009  test 0.7801  rho +0.334  lift 1.3167  (21 min)
  no_degree 98010  test 0.8072  rho +0.176  lift 1.2484  (23 min)
  no_degree 98011  test 0.7835  rho +0.314  lift 1.2588  (26 min)
  no_degree 98012  test 0.7798  rho +0.246  lift 1.1501  (27 min)
  no_degree 98013  test 0.7855  rho +0.301  lift 1.0896  (29 min)
  no_degree 98014  test 0.7899  rho +0.388  lift 1.1488  (31 min)
  no_degree 98

## 6. Sensitivity to the attribution baseline

A zero baseline places the reference outside the space the model was trained on, since no
atom has an all-zero one-hot block. Repeating the measurement from the dataset-average atom
vector, which lies on the manifold, shows whether the choice matters.

In [8]:
BASE_PATH = f"{R}/table68_ig_baseline.csv"
if os.path.exists(BASE_PATH):
    basetab = pd.read_csv(BASE_PATH); print("loaded from checkpoint")
else:
    rows = []
    for enc in ENCODINGS:
        dim = atom_dim(ENCODINGS[enc])
        for baseline in ["zero", "mean"]:
            rh, lf = [], []
            for j in range(50):
                set_seed(99000+j)
                m = build_gnn("gin", HP["hidden"], HP["n_layers"], HP["dropout"],
                              HP.get("residual", True), in_dim=dim)
                sc = grad_scores(m, GRAPHS[enc], "ig", baseline=baseline, ref=MEAN_REF[enc])
                r, _ = rho_with_degree(sc)
                rh.append(r); lf.append(lift(overlap(sc, K_FRAC, 99000+j)))
                del m
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            rows.append({"encoding": enc, "baseline": baseline, "n_draws": 50,
                         "rho_mean": float(np.mean(rh)), "rho_sd": float(np.std(rh, ddof=1)),
                         "lift_mean": float(np.mean(lf))})
            print(f"  {enc:16s} {baseline:5s} rho {rows[-1]['rho_mean']:+.3f}  "
                  f"lift {rows[-1]['lift_mean']:.4f}")
    basetab = pd.DataFrame(rows); basetab.to_csv(BASE_PATH, index=False)
print()
print(basetab.pivot_table(index="encoding", columns="baseline",
                          values=["rho_mean","lift_mean"]).round(4).to_string())

  full             zero  rho +0.669  lift 1.1150
  full             mean  rho +0.538  lift 1.1119
  no_degree        zero  rho +0.470  lift 1.0448
  no_degree        mean  rho +0.491  lift 1.0952
  no_degree_no_h   zero  rho +0.657  lift 1.1203
  no_degree_no_h   mean  rho +0.432  lift 1.0842

               lift_mean         rho_mean        
baseline            mean    zero     mean    zero
encoding                                         
full              1.1119  1.1150   0.5385  0.6688
no_degree         1.0952  1.0448   0.4908  0.4700
no_degree_no_h    1.0842  1.1203   0.4317  0.6574


## 7. What the encoding costs in accuracy

Removing a feature could weaken the model rather than only its attributions. This compares
test performance under the three encodings, so that any change in the association can be read
against any change in what the model learned.

In [9]:
rows = []
for _, r in trained.iterrows():
    rows.append({"encoding": r.encoding, "atom_dim": int(r.atom_dim),
                 "test_roc_auc": r.test_roc_auc,
                 "delta_vs_full": r.test_roc_auc - trained.iloc[0].test_roc_auc,
                 "rho_ig_degree": r.rho_mean,
                 "delta_rho": r.rho_mean - trained.iloc[0].rho_mean,
                 "lift": r.lift_mean})
cost = pd.DataFrame(rows)
cost.to_csv(f"{R}/table69_encoding_cost.csv", index=False)
print(cost.round(4).to_string(index=False))
if len(meta):
    for enc in ["no_degree", "no_degree_no_h"]:
        s = meta[meta.encoding == enc]
        if len(s) < 5: continue
        t, p = stats.ttest_ind(full.test_roc_auc, s.test_roc_auc, equal_var=False)
        tr, pr = stats.ttest_ind(full.spearman_ig_degree, s.rho_ig_degree, equal_var=False)
        print(f"\n  {enc}: accuracy vs full  t = {t:+.2f}, p = {p:.3f}")
        print(f"  {enc}: rho vs full       t = {tr:+.2f}, p = {pr:.4f}")

      encoding  atom_dim  test_roc_auc  delta_vs_full  rho_ig_degree  delta_rho   lift
          full        24        0.7849         0.0000         0.4292     0.0000 1.2655
     no_degree        23        0.7861         0.0011         0.2503    -0.1789 1.1739
no_degree_no_h        22        0.7838        -0.0011         0.4209    -0.0083 1.2781

  no_degree: accuracy vs full  t = -0.41, p = 0.682
  no_degree: rho vs full       t = +4.74, p = 0.0000

  no_degree_no_h: accuracy vs full  t = +0.35, p = 0.730
  no_degree_no_h: rho vs full       t = +0.25, p = 0.8072


## 8. Summary

In [10]:
print("untrained networks")
print(un[["encoding","atom_dim","rho_mean","rho_ci_low","rho_ci_high","lift_mean"]]
        .round(4).to_string(index=False))
print("\ntrained models")
print(trained[["encoding","atom_dim","n_models","test_roc_auc","rho_mean","lift_mean"]]
        .round(4).to_string(index=False))
r0 = float(un[un.encoding == "full"].rho_mean.iloc[0])
r1 = float(un[un.encoding == "no_degree"].rho_mean.iloc[0])
print(f"\nuntrained rho: {r0:+.3f} with the degree feature, {r1:+.3f} without it "
      f"({100*r1/r0:.0f}% retained)")
verdict = ("message passing" if r1 > 0.30 else
           "the input encoding" if r1 < 0.10 else "both paths")
print(f"pre-registered reading: {verdict}")
print("\nWritten to", R)
for f in ["table66_encoding_untrained.csv","table67_encoding_trained.csv",
          "table68_ig_baseline.csv","table69_encoding_cost.csv"]:
    print("  -", f)
import shutil
from google.colab import files
shutil.make_archive("/content/part13_results", "zip", R)
files.download("/content/part13_results.zip")

untrained networks
      encoding  atom_dim  rho_mean  rho_ci_low  rho_ci_high  lift_mean
          full        24    0.6776      0.4396       0.8322     1.1175
     no_degree        23    0.4811      0.0894       0.7061     1.0519
no_degree_no_h        22    0.6577      0.5185       0.7710     1.1175

trained models
      encoding  atom_dim  n_models  test_roc_auc  rho_mean  lift_mean
          full        24        20        0.7849    0.4292     1.2655
     no_degree        23        20        0.7861    0.2503     1.1739
no_degree_no_h        22        20        0.7838    0.4209     1.2781

untrained rho: +0.678 with the degree feature, +0.481 without it (71% retained)
pre-registered reading: message passing

Written to /content/drive/MyDrive/ames_pipeline/results
  - table66_encoding_untrained.csv
  - table67_encoding_trained.csv
  - table68_ig_baseline.csv
  - table69_encoding_cost.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>